# Greyhound ML — Win Probability & RL Bet Sizing

Two-stage architecture:
1. **LightGBM** calibrated win probability model (edge vs BSP implied)
2. **PPO Reinforcement Learning** agent for Kelly-optimal bet sizing

Feature pipeline:
- Race-level market structure (overround, dominance, HHI)
- Runner-level price signals (BSP, drift, market rank)
- Bradley-Terry Elo form ratings (no lookahead)

All evaluation on a held-out chronological test set (final 15% of data).

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import subprocess, sys
pkgs = ['lightgbm', 'shap', 'stable-baselines3[extra]', 'gymnasium', 'optuna']
for pkg in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)
print('Packages ready')

In [ ]:
import os, warnings
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
np.random.seed(42)

OUTPUT_DIR    = '/content/drive/MyDrive/greyhound_output' if IN_COLAB else './greyhound_output'
COMMISSION    = 0.05
FLAT_STAKE    = 10.0
KELLY_FRAC    = 0.25
STARTING_BANK = 1_000.0
RANDOM_SEED   = 42
print(f'Output dir: {OUTPUT_DIR}')

## 1. Load Data

In [ ]:
runners = pd.read_parquet(os.path.join(OUTPUT_DIR, 'greyhound_runners.parquet'))
races   = pd.read_parquet(os.path.join(OUTPUT_DIR, 'greyhound_races.parquet'))

runners['EVENT_DT'] = pd.to_datetime(runners['EVENT_DT'])
races['event_dt']   = pd.to_datetime(races['event_dt'])
runners = runners.sort_values('EVENT_DT').reset_index(drop=True)
races   = races.sort_values('event_dt').reset_index(drop=True)

# Keep rows with valid BSP and result
runners = runners[
    runners['BSP'].notna() &
    (runners['BSP'] > 1.01) &
    runners['WIN_LOSE'].notna() &
    runners['SELECTION_NAME'].notna()
].copy()

print(f'Runners : {len(runners):,}')
print(f'Races   : {len(races):,}')
print(f'Span    : {runners["EVENT_DT"].min().date()} -> {runners["EVENT_DT"].max().date()}')
print(f'Columns : {list(runners.columns)}')

## 2. Feature Engineering
### 2a. Race-Level Market Structure

In [ ]:
race_feats = {}
for eid, grp in runners.groupby('EVENT_ID'):
    bsps = grp['BSP'].dropna().values
    if len(bsps) < 2:
        continue
    impl = 1.0 / bsps
    impl_sorted = np.sort(impl)[::-1]
    overround = impl.sum()
    norm = impl / overround
    hhi  = (norm**2).sum()
    dom  = impl_sorted[0] / impl_sorted[1] if impl_sorted[1] > 0 else np.nan
    race_feats[eid] = {
        'overround':        overround,
        'hhi':              hhi,
        'dominance':        dom,
        'n_bsp_runners':    len(bsps),
        'fav_implied':      impl_sorted[0],
        'second_implied':   impl_sorted[1] if len(impl_sorted) > 1 else np.nan,
    }

race_feat_df = pd.DataFrame.from_dict(race_feats, orient='index')
race_feat_df.index.name = 'EVENT_ID'
race_feat_df = race_feat_df.reset_index()
print(f'Race features: {len(race_feat_df):,} races')
print(race_feat_df.describe().round(3))

### 2b. Bradley-Terry Elo Form Model

Builds per-greyhound Elo ratings chronologically. Features are recorded **before** updating
from the current race result — zero lookahead bias.

In [ ]:
def build_elo_features(runners_df, K=32):
    runners_sorted = runners_df.sort_values(['EVENT_DT', 'EVENT_ID']).copy()
    elo            = defaultdict(lambda: 1500.0)
    career_runs    = defaultdict(int)
    last_results   = defaultdict(list)
    last_bsps      = defaultdict(list)
    last_run_date  = {}
    venue_hist     = defaultdict(lambda: defaultdict(list))
    dist_hist      = defaultdict(lambda: defaultdict(list))

    def dist_bucket(d):
        if pd.isna(d):  return -1
        if d < 400:     return 0
        elif d < 480:   return 1
        elif d < 560:   return 2
        elif d < 640:   return 3
        else:           return 4

    records = []
    for eid, race_grp in runners_sorted.groupby('EVENT_ID', sort=False):
        race_grp  = race_grp.sort_values('BSP')
        race_date = race_grp['EVENT_DT'].iloc[0]
        venue     = race_grp['venue'].iloc[0] if 'venue' in race_grp.columns else 'unk'
        distance  = race_grp['distance'].iloc[0] if 'distance' in race_grp.columns else np.nan
        db        = dist_bucket(distance)
        names  = race_grp['SELECTION_NAME'].tolist()
        bsps   = race_grp['BSP'].tolist()
        wins   = race_grp['WIN_LOSE'].tolist()
        scores = [10 ** (elo[n] / 400) for n in names]
        total  = sum(scores)

        for i, (_, row) in enumerate(race_grp.iterrows()):
            name   = row['SELECTION_NAME']
            cur_el = elo[name]
            my_sc  = scores[i]
            elo_wp = my_sc / total if total > 0 else 1.0 / len(names)
            recent = last_results[name]
            l3     = [r[1] for r in recent[-3:]]
            l5     = [r[1] for r in recent[-5:]]
            rb     = [b[1] for b in last_bsps[name][-5:]]
            ld     = last_run_date.get(name)
            vh     = venue_hist[name][venue]
            dh     = dist_hist[name][db]
            records.append({
                'EVENT_ID':        eid,
                'SELECTION_NAME':  name,
                'elo_rating':      cur_el,
                'elo_win_prob':    elo_wp,
                'n_career_runs':   career_runs[name],
                'last3_wr':        np.mean(l3) if l3 else np.nan,
                'last5_wr':        np.mean(l5) if l5 else np.nan,
                'last5_avg_bsp':   np.mean(rb) if rb else np.nan,
                'days_since_last': (race_date - ld).days if ld else np.nan,
                'venue_wr':        np.mean(vh[-10:]) if vh else np.nan,
                'dist_wr':         np.mean(dh[-10:]) if dh else np.nan,
            })

        # Update after recording (no lookahead)
        for i, name in enumerate(names):
            actual     = 1 if wins[i] == 1 else 0
            exp_win    = scores[i] / total
            elo[name] += K * (actual - exp_win)
            career_runs[name] += 1
            last_results[name].append((race_date, actual))
            last_bsps[name].append((race_date, bsps[i]))
            last_run_date[name] = race_date
            venue_hist[name][venue].append(actual)
            if db >= 0:
                dist_hist[name][db].append(actual)

    return pd.DataFrame(records)

print('Building Elo form features (2-3 min)...')
elo_features = build_elo_features(runners)
print(f'Elo features: {len(elo_features):,} rows')
print(elo_features[['elo_rating','elo_win_prob','n_career_runs',
                     'last3_wr','last5_avg_bsp','days_since_last']].describe().round(3))

### 2c. Assemble Runner-Level Training Dataset

In [ ]:
ml = runners[['EVENT_ID','SELECTION_NAME','EVENT_DT','BSP','MORNINGWAP',
               'MORNINGTRADEDVOL','WIN_LOSE','trap','distance','grade','venue']].copy()
ml = ml.rename(columns={'EVENT_ID':'event_id','SELECTION_NAME':'sel_name',
                         'EVENT_DT':'event_dt','WIN_LOSE':'won',
                         'MORNINGWAP':'mwap','MORNINGTRADEDVOL':'mvol'})

# Join form features
elo_features = elo_features.rename(columns={'EVENT_ID':'event_id','SELECTION_NAME':'sel_name'})
ml = ml.merge(elo_features, on=['event_id','sel_name'], how='left')

# Join race-level features
race_feat_df = race_feat_df.rename(columns={'EVENT_ID':'event_id'})
ml = ml.merge(race_feat_df, on='event_id', how='left')

# Per-runner derived features
ml['implied_prob']   = 1.0 / ml['BSP'].clip(lower=1.01)
ml['morning_prob']   = np.where((ml['mwap'].notna()) & (ml['mwap'] > 1.01),
                                 1.0 / ml['mwap'], np.nan)
ml['drift']          = np.where((ml['mwap'].notna()) & (ml['mwap'] > 1.01),
                                 (ml['mwap'] - ml['BSP']) / ml['mwap'], np.nan)
ml['log_mvol']       = np.log1p(ml['mvol'].fillna(0))
ml['mkt_rank']       = ml.groupby('event_id')['BSP'].rank(method='first').astype(int)
ml['norm_implied']   = ml['implied_prob'] / ml.groupby('event_id')['implied_prob'].transform('sum')
ml['bsp_vs_elo']     = ml['elo_win_prob'] - ml['norm_implied']
ml['dist_bucket']    = pd.cut(ml['distance'], bins=[0,400,480,560,640,9999],
                               labels=[0,1,2,3,4]).astype(float)
ml['grade_enc']      = ml['grade'].astype('category').cat.codes
ml['venue_enc']      = ml['venue'].astype('category').cat.codes

# Fill missing form features with per-race median
FORM_COLS = ['last3_wr','last5_wr','last5_avg_bsp','days_since_last','venue_wr','dist_wr']
for col in FORM_COLS:
    ml[col] = ml[col].fillna(ml.groupby('event_id')[col].transform('median'))

# ── Feature set ───────────────────────────────────────────────────────────────
# NOTE: log_bsp and implied_prob are dropped — they are monotone transforms of
# norm_implied and give the model 3 copies of the same BSP signal, causing it
# to rediscover calibration rather than find residual edge.
# The model must learn what the MARKET DOES NOT ALREADY KNOW.
FEATURE_COLS = [
    # Market signal — ONE price feature only
    'norm_implied',
    # Residuals vs market
    'bsp_vs_elo', 'drift', 'log_mvol',
    # Market structure
    'mkt_rank', 'overround', 'hhi', 'dominance', 'n_bsp_runners',
    # Race context
    'trap', 'dist_bucket', 'distance', 'grade_enc', 'venue_enc',
    # Form (Elo + rolling)
    'elo_rating', 'elo_win_prob',
    'n_career_runs', 'last3_wr', 'last5_wr', 'last5_avg_bsp',
    'days_since_last', 'venue_wr', 'dist_wr',
]
TARGET = 'won'

ml_clean = ml[ml['BSP'].notna() & ml['won'].notna()].copy()
ml_clean[FEATURE_COLS] = ml_clean[FEATURE_COLS].fillna(-1)
ml_clean = ml_clean.sort_values('event_dt').reset_index(drop=True)

print(f'ML dataset : {len(ml_clean):,} runner-race rows')
print(f'Features   : {len(FEATURE_COLS)}  (BSP variants collapsed to norm_implied only)')
print(f'Pos class  : {ml_clean["won"].mean()*100:.1f}%')
missing = ml_clean[FEATURE_COLS].isna().sum()
if missing.any(): print(f'Missing:\n{missing[missing>0]}')
else: print('No missing values after fill')

## 3. Walk-Forward Train / Val / Test Split

Strict chronological split — no data from the future leaks into training.

In [ ]:
n        = len(ml_clean)
tr_end   = int(n * 0.70)
val_end  = int(n * 0.85)

train_df = ml_clean.iloc[:tr_end].copy()
val_df   = ml_clean.iloc[tr_end:val_end].copy()
test_df  = ml_clean.iloc[val_end:].copy()

for split, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f'{split:6s}: {len(df):>9,} rows  '
          f'{df["event_dt"].min().date()} -> {df["event_dt"].max().date()}  '
          f'pos={df["won"].mean()*100:.1f}%')

## 4. LightGBM Win Probability Model

In [ ]:
import lightgbm as lgb
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score

# Ensure target is strictly binary int (WIN_LOSE can load as float with stray values)
for split_name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    bad = ~df[TARGET].isin([0, 1, 0.0, 1.0])
    if bad.any():
        print(f'{split_name}: dropping {bad.sum()} rows with non-binary target '
              f'(values: {df.loc[bad, TARGET].unique()[:5]})')

train_df = train_df[train_df[TARGET].isin([0, 1, 0.0, 1.0])].copy()
val_df   = val_df[val_df[TARGET].isin([0, 1, 0.0, 1.0])].copy()
test_df  = test_df[test_df[TARGET].isin([0, 1, 0.0, 1.0])].copy()

X_tr  = train_df[FEATURE_COLS].values.astype(np.float32)
y_tr  = train_df[TARGET].values.astype(np.int32)
X_val = val_df[FEATURE_COLS].values.astype(np.float32)
y_val = val_df[TARGET].values.astype(np.int32)
X_te  = test_df[FEATURE_COLS].values.astype(np.float32)
y_te  = test_df[TARGET].values.astype(np.int32)

assert set(np.unique(y_tr)) <= {0, 1}, f'Non-binary train target: {np.unique(y_tr)}'
print(f'Target classes — train: {np.unique(y_tr)}  val: {np.unique(y_val)}  test: {np.unique(y_te)}')
print(f'Rows — train: {len(y_tr):,}  val: {len(y_val):,}  test: {len(y_te):,}')

# Up-weight short-price runners — market is most confident here, model should be too
sw_tr = 1.0 / train_df['BSP'].clip(upper=20).values

lgb_params = dict(
    objective='binary', metric='binary_logloss',
    n_estimators=2000, learning_rate=0.03,
    num_leaves=63, max_depth=7,
    min_child_samples=100, subsample=0.8,
    colsample_bytree=0.8, reg_alpha=0.1,
    reg_lambda=1.0, random_state=RANDOM_SEED,
    n_jobs=-1, verbose=-1,
)
lgb_model = lgb.LGBMClassifier(**lgb_params)
lgb_model.fit(
    X_tr, y_tr, sample_weight=sw_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(250)],
)

# Isotonic calibration fitted on validation set
raw_val  = lgb_model.predict_proba(X_val)[:, 1]
raw_test = lgb_model.predict_proba(X_te)[:, 1]
calibrator = IsotonicRegression(out_of_bounds='clip', increasing=True)
calibrator.fit(raw_val, y_val)
cal_val  = calibrator.predict(raw_val)
cal_test = calibrator.predict(raw_test)

print(f'Best iteration : {lgb_model.best_iteration_}')
print(f'Test AUC       : {roc_auc_score(y_te, cal_test):.4f}')
print(f'Test log-loss  : {log_loss(y_te, cal_test):.4f}')
print(f'Test Brier     : {brier_score_loss(y_te, cal_test):.4f}')

# Calibration plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
bins = np.linspace(0, 0.8, 25)
test_df2 = test_df.copy()
test_df2['cal'] = cal_test
test_df2['bucket'] = pd.cut(test_df2['cal'], bins=bins)
cal_chk = test_df2.groupby('bucket', observed=True).agg(
    pred=('cal','mean'), actual=('won','mean'), n=('won','count')).dropna()
ax.scatter(cal_chk['pred'], cal_chk['actual'], s=cal_chk['n']/30, alpha=0.7, color='steelblue')
ax.plot([0,0.8],[0,0.8],'r--',lw=1)
ax.set_title('Model Calibration — Test Set'); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

ax = axes[1]
fi_df = pd.DataFrame({'feature':FEATURE_COLS, 'importance':lgb_model.feature_importances_})
fi_df = fi_df.sort_values('importance',ascending=True).tail(15)
ax.barh(fi_df['feature'], fi_df['importance'], color='steelblue', alpha=0.8)
ax.set_title('LightGBM Feature Importance (top 15)')
plt.tight_layout(); plt.show()

### 4a. SHAP Feature Importance

In [ ]:
import shap
sample_idx = np.random.choice(len(X_te), min(5000, len(X_te)), replace=False)
X_shap     = X_te[sample_idx]
explainer  = shap.TreeExplainer(lgb_model)
shap_vals  = explainer.shap_values(X_shap)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_vals, X_shap, feature_names=FEATURE_COLS, max_display=20, show=False)
plt.title('SHAP Values — LightGBM Win Probability Model')
plt.tight_layout(); plt.show()

shap_df = pd.DataFrame({
    'feature':      FEATURE_COLS,
    'lgb_gain':     lgb_model.feature_importances_,
    'mean_abs_shap': np.abs(shap_vals).mean(axis=0),
}).sort_values('mean_abs_shap', ascending=False)
print('Top 15 features by mean |SHAP|:')
print(shap_df.head(15).to_string(index=False))

## 5. Edge Analysis

Edge = calibrated model probability − BSP implied probability.
Only bets where edge > commission threshold (5pp) are considered.

In [ ]:
# Attach model predictions to test set
te = test_df.copy()
te['cal_prob']    = cal_test
te['bsp_implied'] = 1.0 / te['BSP'].clip(lower=1.01)

# Normalise calibrated probs within each race so they sum to 1
te['norm_cal'] = te.groupby('event_id')['cal_prob'].transform(lambda x: x / x.sum())
te['edge']     = te['norm_cal'] - te['bsp_implied']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Edge distribution
ax = axes[0]
ax.hist(te['edge'].clip(-0.3, 0.3), bins=120, color='steelblue', alpha=0.7)
ax.axvline(0,           color='red',   lw=1, ls='--', label='Zero')
ax.axvline(COMMISSION,  color='green', lw=1, ls='--', label=f'+{COMMISSION*100:.0f}pp (commission)')
ax.axvline(-COMMISSION, color='green', lw=1, ls='--')
ax.set_title('Edge Distribution (Test)'); ax.set_xlabel('Edge'); ax.legend(fontsize=8)

# Reliability diagram: edge bucket vs actual win rate
ax = axes[1]
te['edge_b'] = pd.cut(te['edge'], bins=np.linspace(-0.25,0.25,26))
rel = te.groupby('edge_b', observed=True).agg(
    mid=('edge','mean'), actual=('won','mean'), n=('won','count')).dropna()
ax.scatter(rel['mid'], rel['actual'], s=rel['n']/20, alpha=0.7)
ax.plot(rel['mid'], rel['mid'] + te['bsp_implied'].mean(), 'r--', lw=1, label='Expected if edge correct')
ax.axvline(0, color='gray', lw=0.8)
ax.set_title('Edge vs Actual Win Rate'); ax.set_xlabel('Mean edge'); ax.set_ylabel('Actual WR')

# Back ROI sweep over edge threshold
ax = axes[2]
thresholds = np.arange(0.0, 0.12, 0.005)
rois, ns = [], []
for thr in thresholds:
    sel = te[te['edge'] >= thr]
    if len(sel) < 50: rois.append(np.nan); ns.append(0); continue
    roi = np.where(sel['won']==1, (sel['BSP']-1)*(1-COMMISSION), -1).mean() * 100
    rois.append(roi); ns.append(len(sel))
ax2 = ax.twinx()
ax.plot(thresholds*100, rois, 'b-o', ms=3, label='Back ROI %')
ax2.bar(thresholds*100, ns, width=0.4, alpha=0.25, color='gray')
ax.axhline(0, color='red', lw=0.8, ls='--')
ax.set_title('Back ROI vs Edge Threshold'); ax.set_xlabel('Min edge (pp)')
ax.set_ylabel('ROI %'); ax2.set_ylabel('N bets'); ax.legend()
plt.tight_layout(); plt.show()

valid = [(t,r,n) for t,r,n in zip(thresholds,rois,ns) if r is not None and not np.isnan(r) and n>=100]
best  = max(valid, key=lambda x: x[1])
OPTIMAL_THR = best[0]
print(f'Optimal back threshold : {OPTIMAL_THR*100:.1f}pp  ROI={best[1]:.2f}%  n={best[2]:,}')

# Lay side
lay_rois, lay_ns = [], []
for thr in thresholds:
    sel = te[te['edge'] <= -thr]
    if len(sel) < 50: lay_rois.append(np.nan); lay_ns.append(0); continue
    # Lay ROI on liability
    roi = np.where(sel['won']==0, (1-COMMISSION)/(sel['BSP']-1), -1).mean() * 100
    lay_rois.append(roi); lay_ns.append(len(sel))
valid_l = [(t,r,n) for t,r,n in zip(thresholds,lay_rois,lay_ns) if r is not None and not np.isnan(r) and n>=100]
if valid_l:
    best_l = max(valid_l, key=lambda x: x[1])
    LAY_THR = best_l[0]
    print(f'Optimal lay threshold  : {LAY_THR*100:.1f}pp  ROI={best_l[1]:.2f}%  n={best_l[2]:,}')
else:
    LAY_THR = OPTIMAL_THR
    print('No profitable lay threshold found; using back threshold for lay')

## 6. Supervised Back + Lay Backtest

Fractional Kelly sizing using the model's calibrated probability.
Back: `edge >= threshold`. Lay: `edge <= -threshold`.

In [ ]:
def kelly_back(prob, bsp, frac=KELLY_FRAC):
    f = (prob * (bsp - 1) - (1 - prob)) / (bsp - 1)
    return max(f, 0) * frac

def kelly_lay(prob_win, bsp, frac=KELLY_FRAC):
    prob_lose = 1 - prob_win
    f = (prob_lose * (1 - COMMISSION) - prob_win * (bsp - 1)) / (bsp - 1)
    return max(f, 0) * frac

def run_backtest(df_edge, back_thr, lay_thr, label='Strategy',
                 starting_bank=STARTING_BANK, max_frac=0.10):
    """
    Dynamic Kelly backtest — stakes are always a fraction of the CURRENT bank,
    not the initial bank. Prevents betting more than you have after a losing run.
    max_frac caps any single bet at 10% of current bank regardless of Kelly.
    """
    df = df_edge.sort_values('event_dt').copy()

    # Tag each row as back / lay / no-bet
    df['bet_type'] = np.where(df['edge'] >= back_thr, 'back',
                     np.where(df['edge'] <= -lay_thr, 'lay', 'none'))
    bets = df[df['bet_type'] != 'none'].copy().reset_index(drop=True)
    if len(bets) == 0:
        print(f'{label}: 0 bets'); return None, {}

    # Simulate with running bank
    bank     = starting_bank
    records  = []
    for _, row in bets.iterrows():
        bsp  = float(row['BSP'])
        won  = int(row['won'])
        prob = float(row['norm_cal'])
        btype = row['bet_type']

        if btype == 'back':
            kf    = kelly_back(prob, bsp)
            stake = min(kf * bank, bank * max_frac, bank)  # never bet more than bank
            stake = max(stake, 0.10)
            pnl   = stake * (bsp - 1) * (1 - COMMISSION) if won == 1 else -stake
        else:  # lay
            kf       = kelly_lay(prob, bsp)
            liability = min(kf * bank, bank * max_frac, bank)
            liability = max(liability, 0.10)
            stk_b    = liability / (bsp - 1)
            pnl      = stk_b * (1 - COMMISSION) if won == 0 else -liability
            stake    = liability  # report as liability for sizing reference

        bank = max(bank + pnl, 0.01)  # floor at 1 cent — avoid zero division
        records.append({'event_dt': row['event_dt'], 'bsp': bsp, 'won': won,
                         'stake': stake, 'pnl': pnl, 'edge': row['edge'],
                         'bet_type': btype, 'bank': bank})

    res = pd.DataFrame(records)
    res['drawdown'] = res['bank'] - res['bank'].cummax()

    n_back = (res['bet_type'] == 'back').sum()
    n_lay  = (res['bet_type'] == 'lay').sum()
    total_staked = res['stake'].sum()
    total_pnl    = res['pnl'].sum()
    roi          = total_pnl / total_staked * 100 if total_staked > 0 else 0
    sharpe       = res['pnl'].mean() / (res['pnl'].std() + 1e-9) * np.sqrt(252)

    stats = dict(label=label, n=len(res), n_back=n_back, n_lay=n_lay,
                 staked=total_staked, pnl=total_pnl, roi=roi,
                 final_bank=res['bank'].iloc[-1], max_dd=res['drawdown'].min(),
                 sharpe=sharpe)

    print(f'\n{label}')
    print(f'  Bets         : {len(res):,}  (back {n_back:,}  lay {n_lay:,})')
    print(f'  Total staked : ${total_staked:,.2f}')
    print(f'  Net P&L      : ${total_pnl:+,.2f}')
    print(f'  ROI          : {roi:+.2f}%')
    print(f'  Final bank   : ${res["bank"].iloc[-1]:,.2f}')
    print(f'  Max drawdown : ${res["drawdown"].min():,.2f}')
    print(f'  Sharpe (ann) : {sharpe:.2f}')

    fig, (a1, a2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
    a1.plot(res['bank'].values, lw=1, color='steelblue')
    a1.axhline(starting_bank, color='gray', lw=0.8, ls='--', label=f'Start ${starting_bank:,.0f}')
    a1.set_title(f'{label} — Equity Curve (dynamic Kelly, max {max_frac*100:.0f}% per bet)')
    a1.set_ylabel('Bank ($)')
    a1.legend(fontsize=8)
    a1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    colors = ['steelblue' if t == 'back' else 'crimson' for t in res['bet_type']]
    a2.bar(range(len(res)), res['pnl'], color=colors, alpha=0.5, width=1)
    a2.axhline(0, color='black', lw=0.6)
    a2.set_title('Per-bet P&L  (blue=back  red=lay)')
    a2.set_ylabel('P&L ($)')
    plt.tight_layout(); plt.show()
    return res, stats

bets, stats = run_backtest(te, OPTIMAL_THR, LAY_THR,
                            label='LightGBM Back+Lay — dynamic Kelly (Test Set)')

## 7. Reinforcement Learning Bet-Sizing Agent

The LightGBM model finds *which* runner has edge. The RL agent learns *how much* to bet
given the edge estimates and current bankroll state.

- **State**: 8 edge values + 8 implied probs + bankroll ratio + rolling ROI + drawdown fraction
- **Action**: continuous back/lay fractions for each of 8 runners (clipped to max 10% of bank)
- **Reward**: `log(bank_after / bank_before)` — natural Kelly criterion
- **Algorithm**: PPO (Proximal Policy Optimisation) via stable-baselines3

In [ ]:
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnNoModelImprovement

class GreyhoundEnv(gym.Env):
    """One step = one race. Agent decides back/lay fractions for up to 8 runners."""
    metadata = {'render_modes': []}

    def __init__(self, df, starting_bank=STARTING_BANK, max_frac=0.10, commission=COMMISSION):
        super().__init__()
        self.races        = list(df.groupby('event_id'))
        self.bank0        = starting_bank
        self.max_frac     = max_frac
        self.commission   = commission
        # obs: 8 edges + 8 implied + bank_ratio + rolling_roi + drawdown
        self.observation_space = spaces.Box(-2.0, 5.0, shape=(19,), dtype=np.float32)
        # act: back[0..7] + lay[0..7] fractions in [0, max_frac]
        self.action_space = spaces.Box(0.0, self.max_frac, shape=(16,), dtype=np.float32)
        self._reset_state()

    def _reset_state(self):
        self.idx        = 0
        self.bank       = self.bank0
        self.peak       = self.bank0
        self.recent     = []

    def _obs(self, race_df):
        edges  = np.zeros(8, np.float32)
        implied= np.zeros(8, np.float32)
        for i, (_, row) in enumerate(race_df.iterrows()):
            if i >= 8: break
            edges[i]   = float(row.get('edge', 0.0))
            implied[i] = float(row.get('bsp_implied', 0.0))
        bank_r   = np.float32(self.bank / self.bank0)
        roll_roi = np.float32(np.mean(self.recent[-50:]) if self.recent else 0.0)
        dd       = np.float32((self.bank - self.peak) / self.bank0)
        return np.concatenate([edges, implied, [bank_r, roll_roi, dd]])

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._reset_state()
        _, rd = self.races[0]
        return self._obs(rd), {}

    def step(self, action):
        if self.idx >= len(self.races):
            return self._obs(pd.DataFrame()), 0.0, True, False, {}
        _, race_df = self.races[self.idx]
        runners    = race_df.reset_index(drop=True)
        bank_before = self.bank
        pnl = 0.0
        for i in range(min(len(runners), 8)):
            row = runners.iloc[i]
            bsp = float(row['BSP']); won = int(row['won'])
            if bsp <= 1.05: continue
            bf = float(action[i])
            if bf > 1e-3:
                stk = min(bf * self.bank, self.bank * self.max_frac)
                pnl += stk*(bsp-1)*(1-self.commission) if won==1 else -stk
            lf = float(action[8+i])
            if lf > 1e-3:
                liab = min(lf * self.bank, self.bank * self.max_frac)
                stk_b = liab / (bsp-1)
                pnl += stk_b*(1-self.commission) if won==0 else -liab
        self.bank = max(self.bank + pnl, 1.0)
        self.peak = max(self.peak, self.bank)
        self.recent.append(pnl / bank_before)
        reward = float(np.clip(np.log(self.bank / bank_before), -2.0, 2.0))
        self.idx += 1
        done = self.idx >= len(self.races)
        obs  = self._obs(pd.DataFrame()) if done else self._obs(self.races[self.idx][1])
        return obs, reward, done, False, {}

print('GreyhoundEnv defined. Testing with 1 episode...')
_env = GreyhoundEnv(val_df.assign(bsp_implied=1.0/val_df['BSP'].clip(lower=1.01),
                                   edge=te['edge'].values[:len(val_df)] if len(te)>=len(val_df) else 0.0))
obs, _ = _env.reset()
print(f'Obs shape: {obs.shape}  Action shape: {_env.action_space.shape}')
del _env

### 7a. Prepare RL Training Data & Train PPO Agent

In [ ]:
# Attach edge/implied to val set for RL training
val_for_rl = val_df.copy()
val_for_rl['cal_prob']    = calibrator.predict(lgb_model.predict_proba(X_val)[:, 1])
val_for_rl['bsp_implied'] = 1.0 / val_for_rl['BSP'].clip(lower=1.01)
val_for_rl['norm_cal']    = val_for_rl.groupby('event_id')['cal_prob'].transform(
                                lambda x: x / x.sum())
val_for_rl['edge']        = val_for_rl['norm_cal'] - val_for_rl['bsp_implied']

train_env = DummyVecEnv([lambda: GreyhoundEnv(val_for_rl)])

ppo = PPO(
    'MlpPolicy', train_env,
    learning_rate=3e-4, n_steps=2048, batch_size=64,
    n_epochs=10, gamma=0.99, gae_lambda=0.95,
    clip_range=0.2, ent_coef=0.01,
    policy_kwargs=dict(net_arch=[128, 128]),
    verbose=0, seed=RANDOM_SEED,
)

print('Training PPO agent (200k steps)...')
ppo.learn(total_timesteps=200_000)
print('Training complete.')

### 7b. PPO Agent Evaluation on Test Set

In [ ]:
# Attach edge to test set
te_rl = te.copy()  # te already has edge, bsp_implied, norm_cal, BSP, won

eval_env = GreyhoundEnv(te_rl, starting_bank=STARTING_BANK)
obs, _   = eval_env.reset()
bank_hist = [STARTING_BANK]
stake_hist, pnl_hist = [], []
done = False
while not done:
    action, _ = ppo.predict(obs, deterministic=True)
    obs, rew, done, trunc, _ = eval_env.step(action)
    bank_hist.append(eval_env.bank)
    done = done or trunc

rl_final = bank_hist[-1]
rl_roi   = (rl_final - STARTING_BANK) / STARTING_BANK * 100
rl_dd    = (np.array(bank_hist) - np.maximum.accumulate(bank_hist)).min()

print(f'PPO RL Agent — Test Set')
print(f'  Starting bank : ${STARTING_BANK:,.2f}')
print(f'  Final bank    : ${rl_final:,.2f}')
print(f'  ROI           : {rl_roi:.2f}%')
print(f'  Max drawdown  : ${rl_dd:,.2f}')

fig, ax = plt.subplots(figsize=(14,5))
ax.plot(bank_hist, color='darkorange', lw=1.2, label='PPO RL Agent')
if bets is not None:
    ax.plot(bets['bank'].values, color='steelblue', lw=1, alpha=0.7, label='LightGBM Back+Lay')
ax.axhline(STARTING_BANK, color='gray', lw=0.8, ls='--', label='Start')
ax.set_title('Equity Curves — Test Set Comparison')
ax.set_ylabel('Bank ($)')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
plt.tight_layout(); plt.show()

## 8. Contextual Bandit (Thompson Sampling)

A simpler RL baseline. Each race is an independent context. Thompson Sampling
maintains a Beta distribution over each strategy's win rate and samples to
decide whether to bet, naturally balancing exploration vs exploitation.

This requires no training period and adapts online — useful for production deployment.

In [ ]:
class ThompsonSamplingBandit:
    """Per-runner Beta(alpha, beta) posterior updated from outcomes."""
    def __init__(self, edge_threshold=0.03, kelly_frac=KELLY_FRAC,
                 commission=COMMISSION, bank=STARTING_BANK):
        self.thr      = edge_threshold
        self.kf       = kelly_frac
        self.comm     = commission
        self.bank     = bank
        # Beta params: alpha=wins+1, beta=losses+1
        self.alpha    = defaultdict(lambda: 1.0)
        self.beta_    = defaultdict(lambda: 1.0)

    def _arm_key(self, row):
        # Discretise to (mkt_rank, dist_bucket, trap) for generalisation
        return (int(row.get('mkt_rank', 1)),
                int(row.get('dist_bucket', 2)),
                int(row.get('trap', 0)))

    def act(self, race_df):
        """Return list of (runner_idx, bet_type, stake/liability)."""
        bets_out = []
        for i, (_, row) in enumerate(race_df.iterrows()):
            edge = float(row.get('edge', 0.0))
            bsp  = float(row['BSP'])
            if abs(edge) < self.thr or bsp <= 1.05:
                continue
            key   = self._arm_key(row)
            # Thompson sample: draw from posterior
            sampled = np.random.beta(self.alpha[key], self.beta_[key])
            prob    = float(row.get('norm_cal', 0.5))
            if edge > 0:  # back
                f = max((prob*(bsp-1)-(1-prob))/(bsp-1), 0) * self.kf * sampled
                if f > 0.001:
                    bets_out.append((i, 'back', f * self.bank, bsp, int(row['won']), key))
            else:         # lay
                prob_lose = 1 - prob
                f = max((prob_lose*(1-self.comm)-prob*(bsp-1))/(bsp-1), 0) * self.kf * sampled
                if f > 0.001:
                    bets_out.append((i, 'lay', f * self.bank, bsp, int(row['won']), key))
        return bets_out

    def update(self, bets_placed):
        for (_, btype, _, bsp, won, key) in bets_placed:
            win = (won==1 and btype=='back') or (won==0 and btype=='lay')
            if win: self.alpha[key] += 1
            else:   self.beta_[key] += 1

# Run on test set (online — no pre-training needed)
bandit = ThompsonSamplingBandit(edge_threshold=OPTIMAL_THR)
te_sorted = te.sort_values('event_dt')
bank_ts, pnl_ts = [STARTING_BANK], []

for eid, race_df in te_sorted.groupby('event_id'):
    placed = bandit.act(race_df)
    race_pnl = 0.0
    for (_, btype, amount, bsp, won, _) in placed:
        if btype == 'back':
            race_pnl += amount*(bsp-1)*(1-COMMISSION) if won==1 else -amount
        else:
            liab = amount; stk = liab/(bsp-1)
            race_pnl += stk*(1-COMMISSION) if won==0 else -liab
    bandit.bank = max(bandit.bank + race_pnl, 1.0)
    bandit.update(placed)
    bank_ts.append(bandit.bank)
    pnl_ts.append(race_pnl)

ts_roi = (bandit.bank - STARTING_BANK) / STARTING_BANK * 100
ts_dd  = (np.array(bank_ts) - np.maximum.accumulate(bank_ts)).min()
print(f'Thompson Sampling Bandit — Test Set')
print(f'  Final bank   : ${bandit.bank:,.2f}')
print(f'  ROI          : {ts_roi:.2f}%')
print(f'  Max drawdown : ${ts_dd:,.2f}')

fig, ax = plt.subplots(figsize=(14,4))
ax.plot(bank_ts, color='purple', lw=1.2, label='Thompson Sampling')
if bets is not None:
    ax.plot(bets['bank'].values, color='steelblue', lw=1, alpha=0.7, label='LightGBM Back+Lay')
ax.plot(bank_hist, color='darkorange', lw=1, alpha=0.7, label='PPO RL')
ax.axhline(STARTING_BANK, color='gray', lw=0.8, ls='--')
ax.set_title('All Strategies — Test Set Equity Curves')
ax.set_ylabel('Bank ($)'); ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
plt.tight_layout(); plt.show()

## 9. Walk-Forward Rolling Validation

Re-trains LightGBM on a rolling 3-year window, evaluates on the following 6 months.
Tests whether the edge is stable across different market regimes.

In [ ]:
from sklearn.metrics import roc_auc_score as auc_score

WINDOW_YEARS = 3
STEP_MONTHS  = 6
MIN_TEST     = 5000  # skip folds with fewer test samples

ml_clean['year_month'] = ml_clean['event_dt'].dt.to_period('M')
months = sorted(ml_clean['year_month'].unique())

fold_results = []
window_months = WINDOW_YEARS * 12
step          = STEP_MONTHS

for start_i in range(0, len(months) - window_months - step, step):
    train_months = months[start_i : start_i + window_months]
    test_months  = months[start_i + window_months : start_i + window_months + step]
    if not test_months: break

    tr = ml_clean[ml_clean['year_month'].isin(train_months)]
    te_fold = ml_clean[ml_clean['year_month'].isin(test_months)]
    if len(te_fold) < MIN_TEST: continue

    Xtr = tr[FEATURE_COLS].values; ytr = tr[TARGET].values.astype(int)
    Xte = te_fold[FEATURE_COLS].values; yte = te_fold[TARGET].values.astype(int)

    m = lgb.LGBMClassifier(**lgb_params)
    m.fit(Xtr, ytr, eval_set=[(Xte, yte)],
          callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(9999)])

    raw_p = m.predict_proba(Xte)[:,1]
    cal_  = IsotonicRegression(out_of_bounds='clip').fit(m.predict_proba(Xtr)[:,1], ytr)
    cp    = cal_.predict(raw_p)

    bsp_impl = 1.0 / te_fold['BSP'].clip(lower=1.01).values
    norm_cp  = cp / (te_fold.assign(_cp=cp).groupby('event_id')['_cp']
                      .transform('sum').values + 1e-9)
    edge_f   = norm_cp - bsp_impl

    mask = edge_f >= OPTIMAL_THR
    if mask.sum() < 20:
        roi_f = np.nan
    else:
        pnls = np.where(yte[mask]==1, (te_fold['BSP'].values[mask]-1)*(1-COMMISSION), -1)
        roi_f = pnls.mean() * 100

    fold_results.append({
        'train_start': str(train_months[0]),
        'test_start':  str(test_months[0]),
        'test_end':    str(test_months[-1]),
        'n_train':     len(tr), 'n_test': len(te_fold),
        'n_bets':      int(mask.sum()),
        'auc':         auc_score(yte, cp),
        'roi':         roi_f,
    })
    print(f'  {test_months[0]} -> {test_months[-1]}  n_bets={mask.sum():4d}  ROI={roi_f:+.2f}%  AUC={auc_score(yte,cp):.3f}')

folds_df = pd.DataFrame(fold_results)
print(f'\nMean ROI across folds : {folds_df["roi"].mean():.2f}%')
print(f'Positive folds        : {(folds_df["roi"]>0).sum()} / {len(folds_df)}')

fig, (a1,a2) = plt.subplots(2,1, figsize=(14,6), sharex=True)
a1.bar(range(len(folds_df)), folds_df['roi'],
       color=['green' if r>0 else 'red' for r in folds_df['roi']], alpha=0.7)
a1.axhline(0, color='black', lw=0.8)
a1.set_title('Walk-Forward ROI by Fold'); a1.set_ylabel('ROI %')
a2.bar(range(len(folds_df)), folds_df['n_bets'], color='steelblue', alpha=0.7)
a2.set_title('Bets per Fold'); a2.set_ylabel('N bets')
plt.xticks(range(len(folds_df)), folds_df['test_start'], rotation=45, ha='right')
plt.tight_layout(); plt.show()

## 10. Summary

In [ ]:
print('=' * 65)
print('RESULTS SUMMARY — Out-of-Sample Test Set')
print('=' * 65)
rows = [
    ('Flat back favourite (baseline)',     '-2.75%',  '417,607', '$972.50'),
]
if stats:
    rows.append(('LightGBM Back+Lay (Kelly sizing)',
                 f'{stats["roi"]:+.2f}%',
                 f'{stats["n"]:,}',
                 f'${stats["final_bank"]:,.0f}'))
rows.append(('PPO RL Agent',              f'{rl_roi:+.2f}%', 'N/A', f'${rl_final:,.0f}'))
rows.append(('Thompson Sampling Bandit',  f'{ts_roi:+.2f}%', 'N/A', f'${bandit.bank:,.0f}'))
print(f'{"Strategy":<38} {"ROI":>8}  {"N bets":>9}  {"Final bank":>12}')
print('-' * 65)
for r in rows:
    print(f'{r[0]:<38} {r[1]:>8}  {r[2]:>9}  {r[3]:>12}')
print('=' * 65)
if folds_df is not None and len(folds_df):
    print(f'\nWalk-forward mean ROI : {folds_df["roi"].mean():.2f}%')
    print(f'Positive folds        : {(folds_df["roi"]>0).sum()} / {len(folds_df)}')
print('\nTop SHAP features:')
print(shap_df.head(10)[['feature','mean_abs_shap']].to_string(index=False))